# SQL practice 4: joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-04-joins.ipynb)

15 exercises on outer, semi, anti, self, range and cross joins, and on avoiding fan-out. Each exercise names the [reference entries](sql-reference-04-joins.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 4](sql-reference-04-joins.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

6 easy, 7 medium, 2 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Orders with customer details](#exercise-1-orders-with-customer-details) | Easy | [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join) |
| 2 | [Order counts including zero](#exercise-2-order-counts-including-zero) | Easy | [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join) |
| 3 | [Customers with no orders](#exercise-3-customers-with-no-orders) | Easy | [Semi and anti joins](sql-reference-04-joins.ipynb#semi-and-anti-joins) |
| 4 | [Customers with a paid order, once each](#exercise-4-customers-with-a-paid-order-once-each) | Easy | [Semi and anti joins](sql-reference-04-joins.ipynb#semi-and-anti-joins) |
| 5 | [Paid revenue for every customer](#exercise-5-paid-revenue-for-every-customer) | Medium | [Filters in `ON` versus `WHERE`](sql-reference-04-joins.ipynb#filters-in-on-versus-where) |
| 6 | [Order lines with categories](#exercise-6-order-lines-with-categories) | Easy | [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join) |
| 7 | [Revenue by category](#exercise-7-revenue-by-category) | Medium | [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join) |
| 8 | [Totals without double counting](#exercise-8-totals-without-double-counting) | Medium | [Fan-out: why totals double](sql-reference-04-joins.ipynb#fan-out-why-totals-double) |
| 9 | [Employees and their managers](#exercise-9-employees-and-their-managers) | Easy | [Self joins](sql-reference-04-joins.ipynb#self-joins) |
| 10 | [Paid more than the boss](#exercise-10-paid-more-than-the-boss) | Medium | [Self joins](sql-reference-04-joins.ipynb#self-joins) |
| 11 | [Discount tier by range](#exercise-11-discount-tier-by-range) | Medium | [Non-equi and range joins](sql-reference-04-joins.ipynb#non-equi-and-range-joins) |
| 12 | [Reconcile source and target counts](#exercise-12-reconcile-source-and-target-counts) | Medium | [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join) |
| 13 | [Zero-filled status matrix](#exercise-13-zero-filled-status-matrix) | Medium | [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join) |
| 14 | [Products bought together](#exercise-14-products-bought-together) | Hard | [Self joins](sql-reference-04-joins.ipynb#self-joins), [Non-equi and range joins](sql-reference-04-joins.ipynb#non-equi-and-range-joins) |
| 15 | [Price in effect at order time](#exercise-15-price-in-effect-at-order-time) | Hard | [Non-equi and range joins](sql-reference-04-joins.ipynb#non-equi-and-range-joins) |

## Exercise 1: Orders with customer details

**Difficulty:** Easy · about 3 min · **Practises:** [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join)

*The everyday lookup join.*

Return `order_id`, `name`, `region` and `amount` for shipped orders, ordered by `order_id`.

**Expected output**

| order_id | name | region | amount |
| --- | --- | --- | --- |
| 2 | Acme Corp | North | 84.50 |
| 6 | Cyan Traders | South | 149.00 |
| 9 | Delta Grocers | null | 1180.00 |
| 12 | Foxtrot Foods | West | 96.40 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT o.order_id, c.name, c.region, o.amount
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.status = 'shipped'
    ORDER BY o.order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Order counts including zero

**Difficulty:** Easy · about 5 min · **Practises:** [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join)

*count(*) after a left join counts the NULL row; count(column) does not.*

Return `customer_id`, `name` and `orders` (0 for customers without orders) for every customer, ordered by `orders` descending then `customer_id`.

**Expected output**

| customer_id | name | orders |
| --- | --- | --- |
| 1 | Acme Corp | 3 |
| 2 | Bright Retail | 2 |
| 3 | Cyan Traders | 2 |
| 4 | Delta Grocers | 2 |
| 6 | Foxtrot Foods | 2 |
| 5 | Echo Supplies | 1 |
| 7 | Gamma Sports | 0 |
| 8 | Hotel Homeware | 0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id, c.name, count(o.order_id) AS orders
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY orders DESC, c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Customers with no orders

**Difficulty:** Easy · about 3 min · **Practises:** [Semi and anti joins](sql-reference-04-joins.ipynb#semi-and-anti-joins)

*Anti joins find what is missing.*

Return `customer_id` and `name` of customers that never ordered, ordered by `customer_id`, using an anti join.

**Expected output**

| customer_id | name |
| --- | --- |
| 7 | Gamma Sports |
| 8 | Hotel Homeware |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    LEFT ANTI JOIN orders o ON o.customer_id = c.customer_id
    ORDER BY c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Customers with a paid order, once each

**Difficulty:** Easy · about 5 min · **Practises:** [Semi and anti joins](sql-reference-04-joins.ipynb#semi-and-anti-joins)

*An inner join would repeat customers with several paid orders.*

Return `customer_id` and `name` of customers with at least one paid order, each once, ordered by `customer_id`.

**Expected output**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Bright Retail |
| 3 | Cyan Traders |
| 5 | Echo Supplies |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status = 'paid')
    ORDER BY c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Paid revenue for every customer

**Difficulty:** Medium · about 10 min · **Practises:** [Filters in `ON` versus `WHERE`](sql-reference-04-joins.ipynb#filters-in-on-versus-where)

*Putting the status filter in WHERE silently drops customers.*

Return every customer's `customer_id`, `name` and `paid_revenue` (sum of paid order amounts, 0 if none), ordered by `customer_id`.

**Expected output**

| customer_id | name | paid_revenue |
| --- | --- | --- |
| 1 | Acme Corp | 120.00 |
| 2 | Bright Retail | 260.00 |
| 3 | Cyan Traders | 310.00 |
| 4 | Delta Grocers | 0.00 |
| 5 | Echo Supplies | 74.00 |
| 6 | Foxtrot Foods | 0.00 |
| 7 | Gamma Sports | 0.00 |
| 8 | Hotel Homeware | 0.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id, c.name, coalesce(sum(o.amount), 0) AS paid_revenue
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'paid'
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Order lines with categories

**Difficulty:** Easy · about 5 min · **Practises:** [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join)

*Lookups against an incomplete dimension must not drop facts.*

Return `order_id`, `sku` and `category` (`Unknown` when the product is missing) for orders 11 and 12, ordered by `order_id`, `sku`.

**Expected output**

| order_id | sku | category |
| --- | --- | --- |
| 11 | SKU-4013 | Outdoor |
| 12 | SKU-4005 | Pantry |
| 12 | SKU-4014 | Unknown |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT i.order_id, i.sku, coalesce(p.category, 'Unknown') AS category
    FROM order_items i
    LEFT JOIN products p ON p.sku = i.sku
    WHERE i.order_id IN (11, 12)
    ORDER BY i.order_id, i.sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Revenue by category

**Difficulty:** Medium · about 5 min · **Practises:** [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join)

*Item-level revenue rolled up through a dimension.*

Return `category` (`Unknown` for items without a product) and `revenue` (sum of `qty * unit_price`), highest first.

**Expected output**

| category | revenue |
| --- | --- |
| Pantry | 1340.25 |
| Outdoor | 520.00 |
| Garden | 498.90 |
| Kitchen | 258.50 |
| Unknown | 41.40 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT coalesce(p.category, 'Unknown') AS category, sum(i.qty * i.unit_price) AS revenue
    FROM order_items i
    LEFT JOIN products p ON p.sku = i.sku
    GROUP BY coalesce(p.category, 'Unknown')
    ORDER BY revenue DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Totals without double counting

**Difficulty:** Medium · about 10 min · **Practises:** [Fan-out: why totals double](sql-reference-04-joins.ipynb#fan-out-why-totals-double)

*The most common silent bug in analytics SQL.*

Per `customer_id` with orders, return `revenue` (sum of order amounts) and `units` (sum of item quantities), ordered by `customer_id`. Each order amount must be counted once even though orders have several item rows.

**Expected output**

| customer_id | revenue | units |
| --- | --- | --- |
| 1 | 234.50 | 7 |
| 2 | 315.25 | 4 |
| 3 | 459.00 | 3 |
| 4 | 1269.90 | 21 |
| 5 | 74.00 | 2 |
| 6 | 306.40 | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH units AS (
        SELECT order_id, sum(qty) AS units FROM order_items GROUP BY order_id
    )
    SELECT o.customer_id, sum(o.amount) AS revenue, coalesce(sum(u.units), 0) AS units
    FROM orders o
    LEFT JOIN units u ON u.order_id = o.order_id
    GROUP BY o.customer_id
    ORDER BY o.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Employees and their managers

**Difficulty:** Easy · about 5 min · **Practises:** [Self joins](sql-reference-04-joins.ipynb#self-joins)

*Hierarchies stored as a parent pointer.*

Return `employee` and `manager` (NULL for the top of the hierarchy) for every employee, ordered by `employee`.

**Expected output**

| employee | manager |
| --- | --- |
| Ada | null |
| Brett | Ada |
| Chen | Ada |
| Dana | Brett |
| Eli | Ada |
| Farah | Eli |
| Gus | Eli |
| Hana | Ada |
| Ivan | Hana |
| Jo | Hana |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT e.name AS employee, m.name AS manager
    FROM employees e
    LEFT JOIN employees m ON m.emp_id = e.manager_id
    ORDER BY employee
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Paid more than the boss

**Difficulty:** Medium · about 5 min · **Practises:** [Self joins](sql-reference-04-joins.ipynb#self-joins)

*A classic interview question on self joins.*

Return `employee`, `salary`, `manager` and `manager_salary` for employees earning at least as much as their manager, ordered by `employee`.

**Expected output**

| employee | salary | manager | manager_salary |
| --- | --- | --- | --- |
| Dana | 155000 | Brett | 150000 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT e.name AS employee, e.salary, m.name AS manager, m.salary AS manager_salary
    FROM employees e
    JOIN employees m ON m.emp_id = e.manager_id
    WHERE e.salary >= m.salary
    ORDER BY employee
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Discount tier by range

**Difficulty:** Medium · about 5 min · **Practises:** [Non-equi and range joins](sql-reference-04-joins.ipynb#non-equi-and-range-joins)

*Lookup tables of ranges instead of long CASE expressions.*

Using `tiers` below (inclusive lower bound, exclusive upper bound; NULL upper bound means no limit), return `order_id`, `amount` and `tier`, ordered by `order_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW tiers AS SELECT * FROM VALUES
  ('bronze', 0, 100), ('silver', 100, 300), ('gold', 300, NULL)
AS tiers(tier, min_amount, max_amount)
""")

**Expected output**

| order_id | amount | tier |
| --- | --- | --- |
| 1 | 120.00 | silver |
| 2 | 84.50 | bronze |
| 3 | 30.00 | bronze |
| 4 | 260.00 | silver |
| 5 | 55.25 | bronze |
| 6 | 149.00 | silver |
| 7 | 310.00 | gold |
| 8 | 89.90 | bronze |
| 9 | 1180.00 | gold |
| 10 | 74.00 | bronze |
| 11 | 210.00 | silver |
| 12 | 96.40 | bronze |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT o.order_id, o.amount, t.tier
    FROM orders o
    JOIN tiers t ON o.amount >= t.min_amount AND (o.amount < t.max_amount OR t.max_amount IS NULL)
    ORDER BY o.order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Reconcile source and target counts

**Difficulty:** Medium · about 10 min · **Practises:** [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join)

*Full outer joins compare two sides and show what each is missing.*

Using `source_counts` and `target_counts` below, return `day`, `source_rows`, `target_rows` (0 when missing) and `status`: `ok`, `mismatch`, `missing in target` or `missing in source`. Order by `day`.

In [ ]:
spark.sql("""CREATE OR REPLACE TEMP VIEW source_counts AS SELECT * FROM VALUES
  (DATE'2026-01-01', 100), (DATE'2026-01-02', 120), (DATE'2026-01-03', 90) AS s(day, row_count)""")
spark.sql("""CREATE OR REPLACE TEMP VIEW target_counts AS SELECT * FROM VALUES
  (DATE'2026-01-01', 100), (DATE'2026-01-02', 118), (DATE'2026-01-04', 5) AS t(day, row_count)""")

**Expected output**

| day | source_rows | target_rows | status |
| --- | --- | --- | --- |
| 2026-01-01 | 100 | 100 | ok |
| 2026-01-02 | 120 | 118 | mismatch |
| 2026-01-03 | 90 | 0 | missing in target |
| 2026-01-04 | 0 | 5 | missing in source |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT coalesce(s.day, t.day) AS day,
           coalesce(s.row_count, 0) AS source_rows,
           coalesce(t.row_count, 0) AS target_rows,
           CASE WHEN t.day IS NULL THEN 'missing in target'
                WHEN s.day IS NULL THEN 'missing in source'
                WHEN s.row_count = t.row_count THEN 'ok'
                ELSE 'mismatch' END AS status
    FROM source_counts s
    FULL JOIN target_counts t ON t.day = s.day
    ORDER BY day
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Zero-filled status matrix

**Difficulty:** Medium · about 10 min · **Practises:** [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](sql-reference-04-joins.ipynb#inner-left-right-full-and-cross-join)

*A cross join builds the full grid so missing combinations show as zero.*

For customers 1 to 3 and every status that appears in `orders`, return `customer_id`, `status` and `orders` (0 when none), ordered by `customer_id`, `status`.

**Expected output**

| customer_id | status | orders |
| --- | --- | --- |
| 1 | paid | 1 |
| 1 | placed | 0 |
| 1 | returned | 1 |
| 1 | shipped | 1 |
| 2 | paid | 1 |
| 2 | placed | 1 |
| 2 | returned | 0 |
| 2 | shipped | 0 |
| 3 | paid | 1 |
| 3 | placed | 0 |
| 3 | returned | 0 |
| 3 | shipped | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH statuses AS (SELECT DISTINCT status FROM orders),
         cust AS (SELECT customer_id FROM customers WHERE customer_id BETWEEN 1 AND 3)
    SELECT c.customer_id, s.status, count(o.order_id) AS orders
    FROM cust c
    CROSS JOIN statuses s
    LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = s.status
    GROUP BY c.customer_id, s.status
    ORDER BY c.customer_id, s.status
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Products bought together

**Difficulty:** Hard · about 15 min · **Practises:** [Self joins](sql-reference-04-joins.ipynb#self-joins), [Non-equi and range joins](sql-reference-04-joins.ipynb#non-equi-and-range-joins)

*Market-basket pairs are a common analytics interview question.*

Return `sku_a`, `sku_b` (with `sku_a < sku_b`) and `orders` (number of orders containing both) for every pair that appears together in at least one order, ordered by `orders` descending, then `sku_a`, `sku_b`.

**Expected output**

| sku_a | sku_b | orders |
| --- | --- | --- |
| SKU-4001 | SKU-4007 | 1 |
| SKU-4002 | SKU-4009 | 1 |
| SKU-4003 | SKU-4004 | 1 |
| SKU-4005 | SKU-4014 | 1 |
| SKU-4006 | SKU-4008 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT a.sku AS sku_a, b.sku AS sku_b, count(DISTINCT a.order_id) AS orders
    FROM order_items a
    JOIN order_items b ON b.order_id = a.order_id AND a.sku < b.sku
    GROUP BY a.sku, b.sku
    ORDER BY orders DESC, sku_a, sku_b
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 15: Price in effect at order time

**Difficulty:** Hard · about 15 min · **Practises:** [Non-equi and range joins](sql-reference-04-joins.ipynb#non-equi-and-range-joins)

*Joining facts to a slowly changing dimension by validity interval.*

Using `price_history` below (`valid_to` exclusive, NULL for the current price), return `order_id`, `sku` and `list_price` in effect at the order's `order_ts` for SKU-4003 and SKU-4005 lines, ordered by `order_id`, `sku`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW price_history AS SELECT * FROM VALUES
  ('SKU-4003', 139.00, TIMESTAMP'2025-01-01 00:00:00', TIMESTAMP'2026-01-07 00:00:00'),
  ('SKU-4003', 149.00, TIMESTAMP'2026-01-07 00:00:00', NULL),
  ('SKU-4005', 52.00,  TIMESTAMP'2025-01-01 00:00:00', TIMESTAMP'2026-01-10 00:00:00'),
  ('SKU-4005', 55.25,  TIMESTAMP'2026-01-10 00:00:00', NULL)
AS price_history(sku, list_price, valid_from, valid_to)
""")

**Expected output**

| order_id | sku | list_price |
| --- | --- | --- |
| 4 | SKU-4003 | 139.00 |
| 5 | SKU-4005 | 52.00 |
| 6 | SKU-4003 | 149.00 |
| 12 | SKU-4005 | 55.25 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT i.order_id, i.sku, p.list_price
    FROM order_items i
    JOIN orders o ON o.order_id = i.order_id
    JOIN price_history p
      ON p.sku = i.sku
     AND o.order_ts >= p.valid_from
     AND (o.order_ts < p.valid_to OR p.valid_to IS NULL)
    WHERE i.sku IN ('SKU-4003', 'SKU-4005')
    ORDER BY i.order_id, i.sku
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 4: joins](sql-reference-04-joins.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).